In [30]:
import pandas as pd

In [31]:
evaluations = pd.read_csv("../data/raw/evaluations.csv")
sessions = pd.read_csv("../data/raw/sessions.csv")
model_pricing = pd.read_csv("../data/raw/model_pricing.csv")
llm_calls = pd.read_csv("../data/raw/llm_calls.csv")

In [32]:
df = llm_calls
df.head()

,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no
0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,1338,239,True,906,success,0.002523,2026-08-08 22:00:00,1
1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,1328,413,0,1217,context_overflow,0.000243,1767423822,1
2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,1298,5,1,1127,success,0.001450,2025-12-25 07:14:46,2
3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,1018,67,False,1476,tool_error,0.001029,2026-06-20 11:00:58,1
4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,20,130,1,293,refusal,0.000108,2025-11-15 07:10:09,1


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27854 entries, 0 to 27853
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   call_id            27854 non-null  object 
 1   session_id         27854 non-null  object 
 2   idempotency_key    27854 non-null  object 
 3   model              27854 non-null  object 
 4   tool_name          27854 non-null  object 
 5   prompt_tokens      27854 non-null  object 
 6   completion_tokens  27854 non-null  int64  
 7   cache_hit          27854 non-null  object 
 8   latency_ms         27854 non-null  object 
 9   status             27854 non-null  object 
 10  logged_cost_usd    27160 non-null  float64
 11  called_at          27854 non-null  object 
 12  attempt_no         27854 non-null  int64  
dtypes: float64(1), int64(2), object(10)
memory usage: 2.8+ MB


In [34]:
df.isna().sum()

call_id                0
session_id             0
idempotency_key        0
model                  0
tool_name              0
prompt_tokens          0
completion_tokens      0
cache_hit              0
latency_ms             0
status                 0
logged_cost_usd      694
called_at              0
attempt_no             0
dtype: int64

In [35]:
# df.model.value_counts()

# def value_count(column):
#     print(column.value_counts())

# df.drop(columns=["prompt_tokens", "completion_tokens", "logged_cost_usd"]).apply(value_count)

# Fix column data type and value anomalies

In [36]:
numeric_cols = ["prompt_tokens", "completion_tokens", "logged_cost_usd", "attempt_no", "latency_ms"]
string_cols = df.drop(columns=numeric_cols+["called_at", "cache_hit"]).columns.tolist()
boolean_map = {
    True: True,   False:False,
    "True": True, "False":False,
    "TRUE": True,  "FALSE": False,
    "1"   : True,  "0": False,
    "yes" : True,  "no": False,
    1      : True, 0:False
}

def convert_date_time(column):
    try:
        return pd.to_datetime(column, format="mixed")
    except:
        return pd.to_datetime(column,unit="s")
    

def convert_to_int(column):
    #convert all numeric columns to numeric data

    if column.name in numeric_cols:
        #split latency text to remove "ms from it"
        if column.name == "latency_ms":
            column = column.apply(lambda x: x.split(" ")[0])
        if column.name == "prompt_tokens":
            column = column.apply(lambda x: int(x.replace(",","")))
        try:
            return column.apply(lambda x: Decimal(str(x)) if pd.notna(x) else x)
        except:
            return 

    # resolve model naming conflict wiht capitalisation issues , and
    if column.name in string_cols:
        # first bring all model names to lowercase 
        if column.name == "model":
            column = column.apply(lambda x: x.lower())
        return column.astype("string")

    # handle boolean values
    if column.name == "cache_hit":
        return column.map(boolean_map)

    return column


In [37]:
df= df.apply(convert_to_int)
df["called_at"] = df.called_at.apply(convert_date_time)
df.head()

C:\Users\chris\AppData\Local\Temp\ipykernel_23312\3562644243.py:16: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  return pd.to_datetime(column,unit="s")


,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no
0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,1338,239,True,906,success,0.0025234,2026-08-08 22:00:00,1
1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,1328,413,False,1217,context_overflow,0.000243435,2026-01-03 07:03:42,1
2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,1298,5,True,1127,success,0.0014498,2025-12-25 07:14:46,2
3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,1018,67,False,1476,tool_error,0.001028889,2026-06-20 11:00:58,1
4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,20,130,True,293,refusal,0.000107957,2025-11-15 07:10:09,1


In [38]:
df.isna().sum()

call_id                0
session_id             0
idempotency_key        0
model                  0
tool_name              0
prompt_tokens          0
completion_tokens      0
cache_hit              0
latency_ms             0
status                 0
logged_cost_usd      694
called_at              0
attempt_no             0
dtype: int64

In [39]:
df.to_csv("../data/cleaned/llm_calls.csv")

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27854 entries, 0 to 27853
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   call_id            27854 non-null  string        
 1   session_id         27854 non-null  string        
 2   idempotency_key    27854 non-null  string        
 3   model              27854 non-null  string        
 4   tool_name          27854 non-null  string        
 5   prompt_tokens      27854 non-null  object        
 6   completion_tokens  27854 non-null  object        
 7   cache_hit          27854 non-null  bool          
 8   latency_ms         27854 non-null  object        
 9   status             27854 non-null  string        
 10  logged_cost_usd    27160 non-null  object        
 11  called_at          27854 non-null  datetime64[ns]
 12  attempt_no         27854 non-null  object        
dtypes: bool(1), datetime64[ns](1), object(5), string(6)
memory us

session table has duplicate columns

In [41]:
df

,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no
0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,1338,239,True,906,success,0.0025234,2026-08-08 22:00:00,1
1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,1328,413,False,1217,context_overflow,0.000243435,2026-01-03 07:03:42,1
2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,1298,5,True,1127,success,0.0014498,2025-12-25 07:14:46,2
3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,1018,67,False,1476,tool_error,0.001028889,2026-06-20 11:00:58,1
4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,20,130,True,293,refusal,0.000107957,2025-11-15 07:10:09,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
27849,CL-0017641,SS800659,IK-0017641,mlg-asr-align,none,928,322,False,998,success,0.00017998,2026-02-21 22:08:24,1
27850,CL-0023967,SS804204,IK-0023967,mlg-tutor-sm,lookup_dictionary,1138,170,False,646,success,0.0002616,2026-01-30 12:03:06,1
27851,CL-0010532,SS801424,IK-0010532,mlg-translate-sm,none,834,248,False,160,success,0.000273789,2026-06-20 09:11:00,1
27852,CL-0006531,SS801501,IK-0006531,mlg-tutor-lg,none,108,216,False,5625,timeout,0.001069285,2026-06-12 11:00:00,1


In [42]:
from decimal import Decimal

pricing = {
    row["model"]: {
        "prompt": Decimal(str(row["prompt_usd_per_token"])),
        "completion": Decimal(str(row["completion_usd_per_token"]))
    }
    for _, row in model_pricing.iterrows()
}

In [43]:
import sys
sys.path.insert(0, '../..')
from agent_telimetry.costing import calculate_actual_cost

In [44]:
llm_calls = pd.read_csv("../data/cleaned/llm_calls.csv")
llm_calls.head()

,Unnamed: 0,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no
0,0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,1338,239,True,906,success,0.002523,2026-08-08 22:00:00,1
1,1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,1328,413,False,1217,context_overflow,0.000243,2026-01-03 07:03:42,1
2,2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,1298,5,True,1127,success,0.001450,2025-12-25 07:14:46,2
3,3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,1018,67,False,1476,tool_error,0.001029,2026-06-20 11:00:58,1
4,4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,20,130,True,293,refusal,0.000108,2025-11-15 07:10:09,1


In [45]:
llm_calls['actual_cost'] = llm_calls.apply(lambda row: calculate_actual_cost(
    model=row["model"],
    completion_tokens=row["completion_tokens"],
    input_tokens=row["prompt_tokens"],
    cache_status=row["cache_hit"]
),
axis=1
)

In [55]:
llm_calls.head()

,Unnamed: 0,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no,actual_cost
0,0,CL-0012245,SS800566,IK-0012245,mlg-tutor-lg,explain_grammar,1338,239,True,906,success,0.002523,2026-08-08 22:00:00,1,0
1,1,CL-0013004,SS802118,IK-0013004,mlg-asr-align,fetch_audio,1328,413,False,1217,context_overflow,0.000243,2026-01-03 07:03:42,1,0.0002434199999999999924999721117
2,2,CL-0026948,SS801086,IK-0012655,mlg-tutor-lg,none,1298,5,True,1127,success,0.001450,2025-12-25 07:14:46,2,0
3,3,CL-0008182,SS800963,IK-0008182,mlg-translate-lg,explain_grammar,1018,67,False,1476,tool_error,0.001029,2026-06-20 11:00:58,1,0.001028799999999999953444857446
4,4,CL-0019992,SS804850,IK-0019992,mlg-tutor-sm,none,20,130,True,293,refusal,0.000108,2025-11-15 07:10:09,1,0


In [57]:
retried = llm_calls[llm_calls.duplicated(subset=["idempotency_key"], keep=False)].sort_values(by="idempotency_key", ascending=True)
retried.head()

,Unnamed: 0,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no,actual_cost
22807,22807,CL-0000121,SS801559,IK-0000121,mlg-tutor-sm,translate_span,691,448,False,272,rate_limited,0.000497,2026-02-09 14:02:58,1,0.0004965999999999999775279123327
20055,20055,CL-0027426,SS801559,IK-0000121,mlg-tutor-sm,translate_span,691,448,False,272,success,0.000497,2026-02-09 14:02:58,2,0.0004965999999999999775279123327
298,298,CL-0027171,SS803693,IK-0000143,mlg-translate-lg,none,1047,195,False,8527,success,0.001461,2026-07-01 12:00:58,2,0.001461599999999999933859840245
19613,19613,CL-0000143,SS803693,IK-0000143,mlg-translate-lg,none,1047,195,False,8527,timeout,0.001461,2026-07-01 12:00:58,1,0.001461599999999999933859840245
22828,22828,CL-0000149,SS802858,IK-0000149,mlg-translate-lg,translate_span,762,272,False,1580,rate_limited,0.001480,2026-03-30 08:00:00,1,0.001479999999999999933027205502


In [58]:
retried.status.value_counts()

status
success         900
timeout         477
rate_limited    423
Name: count, dtype: int64

In [66]:
retried.query("tool_name == 'translate_span' and status != 'rate_limited'")

,Unnamed: 0,call_id,session_id,idempotency_key,model,tool_name,prompt_tokens,completion_tokens,cache_hit,latency_ms,status,logged_cost_usd,called_at,attempt_no,actual_cost
20055,20055,CL-0027426,SS801559,IK-0000121,mlg-tutor-sm,translate_span,691,448,False,272,success,0.000497,2026-02-09 14:02:58,2,0.0004965999999999999775279123327
13824,13824,CL-0027513,SS802858,IK-0000149,mlg-translate-lg,translate_span,762,272,False,1580,success,0.001480,2026-03-30 08:00:00,2,0.001479999999999999933027205502
837,837,CL-0000290,SS801638,IK-0000290,mlg-translate-lg,translate_span,413,511,False,9628,timeout,0.001967,2025-12-16 18:03:14,1,0.001965599999999999911052888605
6212,6212,CL-0026960,SS801638,IK-0000290,mlg-translate-lg,translate_span,413,511,False,9628,success,0.001967,2025-12-16 18:03:14,2,0.001965599999999999911052888605
3534,3534,CL-0000681,SS801531,IK-0000681,mlg-tutor-sm,translate_span,854,342,False,10393,timeout,0.000445,2026-05-29 06:01:00,1,0.0004443999999999999798900608955
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23117,23117,CL-0025773,SS804286,IK-0025773,mlg-translate-sm,translate_span,1114,79,True,9961,timeout,0.000215,2026-06-15 13:00:00,1,0
26300,26300,CL-0026197,SS802204,IK-0026197,mlg-tutor-lg,translate_span,537,292,False,8031,timeout,0.001876,2026-02-27 07:01:00,1,0.001875500000000000095653980613
26171,26171,CL-0027373,SS802204,IK-0026197,mlg-tutor-lg,translate_span,537,292,False,8031,success,0.001876,2026-02-27 07:01:00,2,0.001875500000000000095653980613
12318,12318,CL-0026408,SS800702,IK-0026408,mlg-translate-sm,translate_span,1124,5,False,9481,timeout,0.000172,2025-11-17 09:00:00,1,0.0001715999999999999922347759893


- Rate limited calls still have completion_tokens wich should not be the case
- Timeout calls could happen fot multiple reasons so could still be biiled

* remember to create a visualiser for tool sttus and model and status

# evaluation

In [47]:
edf = evaluations

edf.head()

,eval_id,call_id,rubric,verdict,score,evaluator,evaluated_at
0,EV-000001,CL-0008215,hallucination_check,FAIL,0.423,llm-judge,2026-05-03 08:00:00
1,EV-000002,CL-0002992,grammar_correctness,pass,0.963,llm-judge,2026-05-20 09:00:00
2,EV-000003,CL-0022586,translation_accuracy,fail,0.282,llm-judge,2026-05-19 19:00:00
3,EV-000004,CL-0018456,tone_appropriateness,Pass,0.670,human-intern,2026-05-22 12:00:00
4,EV-000005,CL-0014726,hallucination_check,PASS,0.771,llm-judge,2026-05-16 15:00:00


In [48]:
edf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9470 entries, 0 to 9469
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   eval_id       9470 non-null   object 
 1   call_id       9470 non-null   object 
 2   rubric        9470 non-null   object 
 3   verdict       9470 non-null   object 
 4   score         9470 non-null   float64
 5   evaluator     9470 non-null   object 
 6   evaluated_at  9470 non-null   object 
dtypes: float64(1), object(6)
memory usage: 518.0+ KB


In [49]:
cols = ["rubric", "verdict", "evaluator"]

for col in cols:
    print(col)
    edf[col] = edf[col].str.lower()

rubric
verdict
evaluator


In [50]:
edf

,eval_id,call_id,rubric,verdict,score,evaluator,evaluated_at
0,EV-000001,CL-0008215,hallucination_check,fail,0.423,llm-judge,2026-05-03 08:00:00
1,EV-000002,CL-0002992,grammar_correctness,pass,0.963,llm-judge,2026-05-20 09:00:00
2,EV-000003,CL-0022586,translation_accuracy,fail,0.282,llm-judge,2026-05-19 19:00:00
3,EV-000004,CL-0018456,tone_appropriateness,pass,0.670,human-intern,2026-05-22 12:00:00
4,EV-000005,CL-0014726,hallucination_check,pass,0.771,llm-judge,2026-05-16 15:00:00
...,...,...,...,...,...,...,...
9465,EV-009466,CL-0017996,grammar_correctness,pass,0.923,human-intern,2026-05-02 11:00:00
9466,EV-009467,CL-0006489,safety,pass,0.706,llm-judge,2026-05-24 12:00:00
9467,EV-009468,CL-0007004,translation_accuracy,pass,0.945,human-intern,2026-05-12 08:00:00
9468,EV-009469,CL-0001081,safety,pass,0.871,llm-judge,2026-05-07 10:00:00
